# Day 4 — The Agent

Day 3 gave me search. But *I* had to decide what to search for, run it, and
paste results into a prompt. Today the LLM does that itself: I hand it a
**search tool** and it decides when to call it, what to query, and when it has
enough to answer.

That's the difference between a search box and an agent. A plain RAG pipeline
does exactly one search with the user's words. An agent can decide to search
several times — decomposing "how do I automate dbt?" into scheduling, then CI,
then orchestration — and synthesize across them.

**Stack note:** I'm using the Anthropic API (not the course's OpenAI). Tool use
has a slightly different shape than OpenAI function calling — noted inline.

In [25]:
# uv add minsearch sentence-transformers anthropic python-dotenv numpy

## 1. Rebuild the search backend from Day 3

Load chunks, build both indexes. This is Day 3's work condensed into the setup
the agent needs. Embeddings were cached for scoped results only, so I will re-encode all dbt chunks, so the result - the embeddings are reusable. I also had to change the embedding model (as per next cell - very slow execution with model "multi-qa-distilbert-cos-v1")

In [26]:
import json
import os
import numpy as np

from minsearch import Index, VectorSearch

dbt_chunks = []
with open("chunks.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        dbt_chunks.append(json.loads(line))

# lexical
index = Index(text_fields=["chunk"], keyword_fields=["filename"])
index.fit(dbt_chunks)

# vector
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer("multi-qa-distilbert-cos-v1")

if os.path.exists("embeddings.npy"):
    embeddings = np.load("embeddings.npy")
else:
    texts = [c["chunk"] for c in dbt_chunks]
    embeddings = embedding_model.encode(texts, batch_size=64, show_progress_bar=True)
    np.save("embeddings.npy", embeddings)

vindex = VectorSearch(keyword_fields=[])
vindex.fit(embeddings, dbt_chunks)

print(f"indexes ready over {len(dbt_chunks)} chunks")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

indexes ready over 7910 chunks


Checking encoding time needed with "multi-qa-distilbert-cos-v1"

In [27]:
import torch, time
print("threads:", torch.get_num_threads())
print("cuda:", torch.cuda.is_available())

# time encoding just 100 chunks
t = time.time()
embedding_model.encode(texts[:100], batch_size=32)
print(f"100 chunks in {time.time()-t:.1f}s")

threads: 10
cuda: False
100 chunks in 9.8s


## 2. The search function the agent will call

Hybrid search from Day 3 — lexical + vector, merged and deduped. The agent
never sees this code; it only sees the *description* I attach in the tool
schema. Good tool descriptions are prompt engineering: the model decides
whether to call it based entirely on the description.

I return `filename` alongside each chunk so the agent can cite sources.

In [28]:
def hybrid_search(query, num_results=5):
    lex = index.search(query, num_results=num_results)
    vec = vindex.search(embedding_model.encode(query), num_results=num_results)

    seen, merged = set(), []
    for r in lex + vec:
        key = (r["filename"], r["chunk"][:50])
        if key not in seen:
            seen.add(key)
            merged.append(r)
    return merged[:num_results]


def text_search(query, num_results=5):
    """The tool body. Returns a JSON-serializable list for the agent."""
    results = hybrid_search(query, num_results=num_results)
    return [{"filename": r["filename"], "chunk": r["chunk"]} for r in results]


# smoke test
hits = text_search("how do I schedule a dbt job")
print(f"{len(hits)} hits, first: {hits[0]['filename']}")

5 hits, first: website/docs/guides/manual-install-qs.md


## 3. Describe the tool to the agent

This is the schema Claude receives. Three parts:
- `name` — what the model calls
- `description` — WHEN to call it (this is the important part; the model reads
  this to decide). I explicitly tell it that multiple searches are allowed.
- `input_schema` — JSON Schema for the arguments

OpenAI wraps this in `{"type": "function", "function": {...}}`; Anthropic uses
the flat shape below.

In [29]:
TOOLS = [
    {
        "name": "text_search",
        "description": (
            "Search the dbt documentation for relevant passages. Call this "
            "whenever you need factual information about dbt to answer the user's "
            "question. You may call it MULTIPLE times with different queries to "
            "gather complete information — for broad questions, search each "
            "sub-topic separately. Returns a list of documentation chunks with "
            "their source filenames."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "The search query. Use natural language.",
                }
            },
            "required": ["query"],
        },
    }
]

## 4. The agentic loop

The core mechanic. Claude's response has a `stop_reason`:
- `"tool_use"` → it wants to call a tool. I run the tool, append the result,
  and loop back so it can search again or answer.
- anything else (`"end_turn"`) → it's done; return the text.

This loop is what makes it an *agent* — the model drives, deciding how many
searches to run. `max_turns` is a safety cap so a confused agent can't loop
forever.

**Message-shape gotchas vs. OpenAI:**
- The assistant's tool-call turn is appended as `{"role":"assistant","content":resp.content}` — the raw content blocks, not a string.
- Tool results go back as a `user` turn containing `tool_result` blocks, each
  with the matching `tool_use_id`.

In [30]:
from dotenv import load_dotenv
import anthropic

load_dotenv()
client = anthropic.Anthropic()

TOOL_FUNCTIONS = {"text_search": text_search}

SYSTEM_PROMPT = (
    "You are a dbt documentation assistant. Answer questions using the "
    "text_search tool to ground every answer in the docs. Search as many times "
    "as needed. Cite the source filename for each claim. If the docs don't "
    "cover something, say so rather than guessing."
)


def run_agent(question, max_turns=6, verbose=True):
    messages = [{"role": "user", "content": question}]

    for turn in range(max_turns):
        resp = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=2048,
            system=SYSTEM_PROMPT,
            tools=TOOLS,
            messages=messages,
        )

        # not a tool call -> the agent is answering; we're done
        if resp.stop_reason != "tool_use":
            answer = "".join(b.text for b in resp.content if b.type == "text")
            if verbose:
                print(f"[turn {turn}] final answer\n")
            return answer

        # record the assistant's tool-call turn (raw content blocks)
        messages.append({"role": "assistant", "content": resp.content})

        # run each requested tool, collect results
        tool_results = []
        for block in resp.content:
            if block.type == "tool_use":
                if verbose:
                    print(f"[turn {turn}] search: {block.input.get('query')!r}")
                output = TOOL_FUNCTIONS[block.name](**block.input)
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": json.dumps(output),
                })

        # feed results back as a user turn, loop
        messages.append({"role": "user", "content": tool_results})

    return "Stopped: reached max turns without a final answer."

## 5. Run it — a simple question

Watch the trace: the agent decides to search, gets chunks, then answers.

In [31]:
print(run_agent("How do I schedule a dbt job to run on a cron schedule?"))

[turn 0] search: 'schedule dbt job cron schedule'
[turn 0] search: 'job scheduling cron'
[turn 1] final answer

Based on the dbt documentation, here's how to schedule a dbt job to run on a cron schedule:

## Using dbt Cloud (Recommended)

According to the documentation, **dbt Cloud is the easiest and most reliable way** to schedule and deploy jobs. To set up a cron schedule in dbt Cloud:

1. **Create a Deploy Job**: From your deployment environment page, click **Create job** and select **Deploy job**.

2. **Configure the Cron Schedule**: In the **Triggers** section, select:
   - **Run on schedule**
   - Under **Timing**, choose **Cron schedule**
   - Enter your cron syntax

### Cron Schedule Details

**Important notes** ([source: deploy-jobs.md]()):
- Cron schedules in dbt Cloud use **UTC** and don't convert to your local timezone or adjust for daylight saving time
- **Frequency restrictions vary by plan**:
  - Developer plans: Minimum interval of every 10 minutes
  - Starter, Enterpri

## 6. Run it — the broad question that stumped plain RAG

Back on Day 3 I noted "how do I automate things in dbt?" is near-impossible for
single-shot retrieval: ~774 relevant chunks across 95 doc areas, and search
returns 5. The agent should handle this better by searching several sub-topics.
Watch how many times it searches.

In [32]:
print(run_agent("How can I automate things in dbt?"))

[turn 0] search: 'automation in dbt'
[turn 0] search: 'automate dbt workflows'
[turn 0] search: 'scheduling dbt runs'
[turn 1] search: 'dbt Cloud jobs scheduler'
[turn 1] search: 'dbt API orchestration triggering'
[turn 2] final answer

Based on the dbt documentation, here are the main ways you can automate things in dbt:

## **1. Built-in dbt Cloud Job Scheduler**

The [dbt Cloud Job Scheduler](website/docs/docs/deploy/job-scheduler.md) is the primary automation tool within dbt. It supports multiple execution models:

- **Cron-based execution**: Run dbt jobs on a predetermined schedule
- **Event-driven execution**: 
  - Trigger based on job completion (jobs triggering other jobs)
  - Trigger when pull requests are merged (merge jobs)
  - Trigger via API
  - Manual "Run now" triggering

The scheduler handles queuing, creating environments, logging, and storing artifacts automatically.

## **2. External Orchestration Tools**

If you want to use external tools, [dbt supports integration 

## 7. Run it — the Wizard question from Day 3

The one where lexical failed and vector found the answer at rank 0. The agent
has hybrid search, so it should nail this — and now it decides the query wording
itself rather than using my literal phrasing.

In [33]:
print(run_agent(
    "Can I use Wizard to create a new model out of an existing SQL query, "
    "based on source tables?"
))

[turn 0] search: 'Wizard create model from SQL query source tables'
[turn 0] search: 'dbt Wizard feature functionality'
[turn 1] search: 'Wizard building models creating new models'
[turn 1] search: 'Wizard convert SQL to dbt model'
[turn 2] search: 'Wizard intro best practices building new models how to use'
[turn 2] search: 'Wizard Studio IDE create model from query'
[turn 3] search: 'Wizard build refactor models from natural language SQL'
[turn 4] final answer

Perfect! Based on the documentation, I now have a clear answer. Let me provide you with the information:

Yes, you can use dbt Wizard to create a new model from an existing SQL query based on source tables. Here's what the documentation says:

**In the Studio IDE:**
According to the dbt documentation, dbt Wizard in the Studio IDE allows you to **"write or refactor dbt models from natural language"** ([wizard-ide.md](website/docs/docs/dbt-ai/wizard-ide.md)). You can describe what you want to create or change, and Wizard will g

## 8. Package it as a reusable function

For Day 6 (Streamlit) I'll need a clean entry point: question in, answer out,
no printing. This is that.

In [34]:
def ask_dbt(question):
    """Single entry point for the app: question -> grounded answer string."""
    return run_agent(question, verbose=False)


# quick check
ask_dbt("What is an incremental model?")

"Based on the dbt documentation, here's what an incremental model is:\n\n## What is an Incremental Model?\n\nAn **incremental model** is a type of [materialization](website/docs/docs/build/materializations.md) in dbt that allows dbt to insert or update records into a table since the last time that model was run, rather than rebuilding the entire table from scratch each time.\n\n### Key Characteristics\n\nAccording to the documentation, incremental models are defined with `select` statements and require you to configure:\n\n1. **How to filter rows on an incremental run** - which new/updated records to include\n2. **The unique key of the model** (if any) - to identify which records to update\n\n### When to Use Incremental Models\n\nIncremental models are best suited for:\n- **Event-style data** - data that represents discrete events or transactions\n- **Large datasets** where full refreshes are too slow - they should be adopted when your `dbt run`s are becoming too slow, not as a startin

## Day 4 findings

1. **The agent decides when and how often to search** — the whole shift from
   Day 3. I no longer craft the query or the number of searches; the model does,
   based on the tool description.
2. **Tool description = prompt engineering.** Telling the agent explicitly that
   it *may search multiple times* is what makes it decompose broad questions.
   Without that line it tends to search once and stop.
3. **Broad questions improved most.** "How do I automate dbt?" — hopeless for
   single-shot RAG on Day 3 — becomes tractable because the agent runs several
   targeted searches and synthesizes. (Check the trace above for how many.)
4. **Anthropic tool-use shape** differs from OpenAI: flat tool schema, assistant
   turn carries raw content blocks, results return as `tool_result` blocks in a
   user turn keyed by `tool_use_id`.
5. **`max_turns` matters** — without a cap, a confused agent can loop forever
   (and burn tokens). A safety limit is not optional in production.

**Next (Day 5):** evaluation. Right now I'm eyeballing answers. Day 5 builds a
question set and measures whether retrieval surfaces the right chunk and whether
the agent's answer is grounded — turning "looks good" into a number.

In [35]:
import json

dbt_chunks = []
with open("chunks.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        dbt_chunks.append(json.loads(line))

print(len(dbt_chunks), "dbt_chunks")
print("embeddings:", len(embeddings))
# these must be equal

7910 dbt_chunks
embeddings: 7910
